# Knowledge Tracing: per-student topic mastery (BKT)

Fits a **BKT** (Bayesian Knowledge Tracing) model over this project's student interaction
graph to answer the actual product question: *for a given student, which topics are they
weak in?*

BKT is a 2-state HMM per (student, leaf topic), updated online per attempt — cheap,
interpretable, and it needs only a handful of attempts per pair to produce a defensible
`p_know`, including graceful cold-start behavior (falls back to the prior) for a new
student or a just-published topic. Its output is directly the personalization signal
this system needs: one `p_know` per (student, topic), so "what is this student weak in"
is just "sort their topics by `p_know` ascending."

Data: `scripts/generate_dummy_interactions.py`'s synthetic students, generated against
the real question bank's subject/topic taxonomy (`notebooks/mcq_output/question_bank.json`)
so topic-level sequences here have the same shape production data will have. Current scale:
20 students, ~1100 quiz answers, 56 simulated topics.

Requires `make neo4j-up` and the dummy data already generated (see
`docs/07-operations.md` / this repo's README).

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score, log_loss

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.student_kg.driver import make_driver

pd.set_option("display.max_rows", 20)
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

## 1. Pull interaction sequences from Neo4j

One row per `QUIZ_ANSWER` event, joined to the leaf `Topic` it's tagged with via
`(:Question)-[:BELONGS_TO]->(:Topic)` (see `src/quiz/attempts.py::record_attempt` — the
leaf topic is the deepest element of `topic_tag`). Ordered by `ts` within each student so
sequence models see the real chronological order.

In [2]:
_SEQUENCES_QUERY = """
MATCH (s:Student)-[:ATTEMPTED]->(:QuizSession)-[:HAS_ANSWER]->(e:InteractionEvent {type: "QUIZ_ANSWER"})
MATCH (e)-[:FOR_QUESTION]->(q:Question)-[:BELONGS_TO]->(t:Topic)
RETURN s.id AS student_id, t.path AS topic_path, e.question_uid AS question_uid,
       e.correct AS correct, e.confidence AS confidence,
       e.time_taken_seconds AS time_taken_seconds, e.ts AS ts
ORDER BY s.id, e.ts ASC
"""

driver = make_driver()
with driver.session() as session:
    records = [dict(r) for r in session.run(_SEQUENCES_QUERY)]
driver.close()

df = pd.DataFrame(records)
df["ts"] = df["ts"].apply(lambda z: z.to_native())
df["correct"] = df["correct"].astype(int)
print(f"{len(df)} attempts, {df.student_id.nunique()} students, {df.topic_path.nunique()} topics")
df.head()

1102 attempts, 20 students, 56 topics


,student_id,topic_path,question_uid,correct,confidence,time_taken_seconds,ts
0,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,master_mcq::copy_of_paket_a46_pdf::0005,1,unsure,26.67,2026-03-11 00:01:48.521909+00:00
1,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,master_mcq::salinan_paket_c22_pdf::0005,1,unsure,23.33,2026-03-11 00:03:26.594681+00:00
2,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,master_mcq::paket_42_pdf::0003,1,unsure,23.01,2026-03-11 00:04:46.879666+00:00
3,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,master_mcq::copy_of_paket_a1_pdf::0005,1,confident,11.15,2026-03-11 00:08:42.256410+00:00
4,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,master_mcq::copy_of_paket_a46_pdf::0005,0,unsure,9.80,2026-03-11 00:10:07.918302+00:00


In [3]:
# Data-scale diagnostics up front — these numbers are what make the BKT-vs-AKT call later.
attempts_per_student_topic = df.groupby(["student_id", "topic_path"]).size()
print("attempts per (student, topic) pair:")
print(attempts_per_student_topic.describe())
print()
print("median attempts/student/topic:", attempts_per_student_topic.median())
print("pairs with >=5 attempts:", (attempts_per_student_topic >= 5).sum(),
      "/", len(attempts_per_student_topic))

attempts per (student, topic) pair:
count    184.000000
mean       5.989130
std        2.427054
min        3.000000
25%        4.000000
50%        6.000000
75%        8.000000
max       19.000000
dtype: float64

median attempts/student/topic: 6.0
pairs with >=5 attempts: 128 / 184


## 2. BKT baseline

Standard 2-state (knows / doesn't-know) HMM per (student, topic):

- `p_init` — prior P(knows) before any evidence.
- `p_transit` — P(learns) between an unknown state and the next attempt.
- `p_slip` — P(wrong | knows).
- `p_guess` — P(correct | doesn't know).

Update rule (per attempt, standard BKT posterior + learning-transition step):

```
p_correct_given_know    = 1 - p_slip
p_correct_given_not_know = p_guess

if observed correct:
    p_know_post = p_know * p_correct_given_know / (p_know * p_correct_given_know + (1-p_know) * p_correct_given_not_know)
else:
    p_know_post = p_know * p_slip / (p_know * p_slip + (1-p_know) * (1-p_guess))

p_know_next = p_know_post + (1 - p_know_post) * p_transit
```

Global params here are fixed defaults (not per-topic fit via EM) — reasonable for a first
pass; EM-fitting `p_slip`/`p_guess`/`p_transit` per topic is listed as follow-up work at
the end.

In [4]:
BKT_PARAMS = dict(p_init=0.3, p_transit=0.1, p_slip=0.1, p_guess=0.25)


def bkt_update(p_know: float, correct: bool, params: dict) -> float:
    p_slip, p_guess, p_transit = params["p_slip"], params["p_guess"], params["p_transit"]
    if correct:
        num = p_know * (1 - p_slip)
        denom = num + (1 - p_know) * p_guess
    else:
        num = p_know * p_slip
        denom = num + (1 - p_know) * (1 - p_guess)
    p_know_post = num / denom if denom > 0 else p_know
    return p_know_post + (1 - p_know_post) * p_transit


def bkt_predict_proba(p_know: float, params: dict) -> float:
    """P(correct) implied by current p_know, before seeing the observation."""
    return p_know * (1 - params["p_slip"]) + (1 - p_know) * params["p_guess"]


def run_bkt(df: pd.DataFrame, params: dict = BKT_PARAMS) -> tuple[pd.DataFrame, dict]:
    """Runs BKT forward over every (student, topic) sequence in chronological order.
    Returns per-attempt predicted P(correct) (predicted BEFORE the update, i.e. a genuine
    next-step prediction, not a fitted-in-hindsight one) and the final p_know per pair."""
    preds = np.empty(len(df))
    state: dict[tuple[str, str], float] = {}
    for i, row in enumerate(df.itertuples()):
        key = (row.student_id, row.topic_path)
        p_know = state.get(key, params["p_init"])
        preds[i] = bkt_predict_proba(p_know, params)
        state[key] = bkt_update(p_know, bool(row.correct), params)
    out = df.copy()
    out["bkt_pred"] = preds
    return out, state


df_sorted = df.sort_values(["student_id", "topic_path", "ts"]).reset_index(drop=True)
bkt_results, bkt_final_state = run_bkt(df_sorted)
bkt_results[["student_id", "topic_path", "ts", "correct", "bkt_pred"]].head(10)

,student_id,topic_path,ts,correct,bkt_pred
0,0597911b-0e08-477c-95a2-6f2eda2ccc56,Abdominal wall - hollow organ - organs > Abdom...,2026-03-18 11:06:09.809515+00:00,0,0.445000
1,0597911b-0e08-477c-95a2-6f2eda2ccc56,Abdominal wall - hollow organ - organs > Abdom...,2026-03-18 11:07:14.383310+00:00,1,0.346622
2,0597911b-0e08-477c-95a2-6f2eda2ccc56,Abdominal wall - hollow organ - organs > Abdom...,2026-03-18 11:09:13.214882+00:00,0,0.540789
3,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 01 2024 rc > Serum Enzyme Measurement,2026-04-01 08:45:26.962994+00:00,0,0.445000
4,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 01 2024 rc > Serum Enzyme Measurement,2026-04-01 08:48:58.530190+00:00,0,0.346622
5,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 01 2024 rc > Serum Enzyme Measurement,2026-04-01 08:52:12.831354+00:00,1,0.328309
6,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,2026-03-11 00:01:48.521909+00:00,1,0.445000
7,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,2026-03-11 00:03:26.594681+00:00,1,0.669944
8,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,2026-03-11 00:04:46.879666+00:00,1,0.822736
9,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,2026-03-11 00:08:42.256410+00:00,1,0.878870


In [5]:
bkt_auc = roc_auc_score(bkt_results["correct"], bkt_results["bkt_pred"])
bkt_ll = log_loss(bkt_results["correct"], bkt_results["bkt_pred"].clip(1e-4, 1 - 1e-4))
print(f"BKT next-step prediction — AUC: {bkt_auc:.4f}, log loss: {bkt_ll:.4f}")

BKT next-step prediction — AUC: 0.6638, log loss: 0.6661


## 3. Per-student weak-topic view

This is the shape a `MASTERS` edge (`Student -> Topic`, `p_know`) would take in the graph
— see the design note at the end for wiring this into `src/quiz/attempts.py::record_attempt`.

Each student gets their own ranked list — this is the personalization output: not one
global model output, but one `p_know` value per (student, topic) pair, so weak topics are
specific to that student's own history and never averaged across the cohort.

In [6]:
mastery_rows = [
    {"student_id": sid, "topic_path": tp, "p_know": pk}
    for (sid, tp), pk in bkt_final_state.items()
]
mastery_df = pd.DataFrame(mastery_rows)

n_observations = (
    df_sorted.groupby(["student_id", "topic_path"]).size().rename("n_observations")
)
mastery_df = mastery_df.join(n_observations, on=["student_id", "topic_path"])
mastery_df.sort_values(["student_id", "p_know"]).head(10)

,student_id,topic_path,p_know,n_observations
0,0597911b-0e08-477c-95a2-6f2eda2ccc56,Abdominal wall - hollow organ - organs > Abdom...,0.187679,3
5,0597911b-0e08-477c-95a2-6f2eda2ccc56,karbohidrat > Carbohydrate digestion enzymes,0.187679,3
4,0597911b-0e08-477c-95a2-6f2eda2ccc56,Embriologi Pencernaan > Cleft palate causes,0.387789,5
1,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 01 2024 rc > Serum Enzyme Measurement,0.397236,3
3,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme isozymes classif...,0.508666,4
2,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme activity and sub...,0.946707,5
13,06a0b289-5aeb-4e83-9106-6838034d5f96,karbohidrat > Carbohydrate metabolism,0.115401,5
10,06a0b289-5aeb-4e83-9106-6838034d5f96,Endocrine 1-25 > Antidiuretic hormone function,0.116596,7
7,06a0b289-5aeb-4e83-9106-6838034d5f96,ENZIME FK 01 2024 rc > enzyme catalysis mechanism,0.116717,8
11,06a0b289-5aeb-4e83-9106-6838034d5f96,Endocrine 1-25 > Hormone structure classification,0.127476,7


In [7]:
def weakest_topics(
    student_id: str,
    mastery_df: pd.DataFrame,
    top_n: int = 5,
    min_observations: int = 3,
) -> pd.DataFrame:
    """This student's lowest-p_know topics, ascending — the recommend-what-to-study query
    the personalization system runs (GET /students/me/mastery, sorted client- or
    server-side).

    Adds a `low_evidence` flag for pairs with fewer than `min_observations` attempts: with
    only 1-2 attempts, p_know is still close to p_init (the prior) rather than a real
    read on the student, so surfacing it as "weak" without qualification would overstate
    confidence. Rows are NOT dropped — a topic never attempted is still worth surfacing as
    "unknown, go try it" — just labeled so the caller (or UI) can render it differently
    (e.g. "not enough data yet" instead of a confident weak-topic claim)."""
    out = (
        mastery_df[mastery_df["student_id"] == student_id]
        .sort_values("p_know")
        .head(top_n)
        .reset_index(drop=True)
    )
    out["low_evidence"] = out["n_observations"] < min_observations
    return out


example_student = mastery_df["student_id"].iloc[0]
print(f"weakest topics for student {example_student}:")
weakest_topics(example_student, mastery_df)

weakest topics for student 0597911b-0e08-477c-95a2-6f2eda2ccc56:


,student_id,topic_path,p_know,n_observations,low_evidence
0,0597911b-0e08-477c-95a2-6f2eda2ccc56,Abdominal wall - hollow organ - organs > Abdom...,0.187679,3,False
1,0597911b-0e08-477c-95a2-6f2eda2ccc56,karbohidrat > Carbohydrate digestion enzymes,0.187679,3,False
2,0597911b-0e08-477c-95a2-6f2eda2ccc56,Embriologi Pencernaan > Cleft palate causes,0.387789,5,False
3,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 01 2024 rc > Serum Enzyme Measurement,0.397236,3,False
4,0597911b-0e08-477c-95a2-6f2eda2ccc56,ENZIME FK 02 2024 rc > Enzyme isozymes classif...,0.508666,4,False


In [8]:
sid = example_student
weak = weakest_topics(sid, mastery_df, top_n=3)
for tp in weak["topic_path"]:
    sub = df_sorted[(df_sorted.student_id == sid) & (df_sorted.topic_path == tp)]
    print(tp, "acc:", sub["correct"].mean(), "n:", len(sub))

Abdominal wall - hollow organ - organs > Abdominal wall hollow organ organs acc: 0.3333333333333333 n: 3
karbohidrat > Carbohydrate digestion enzymes acc: 0.3333333333333333 n: 3
Embriologi Pencernaan > Cleft palate causes acc: 0.2 n: 5
